In [ ]:
import asyncio
import json
import os
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    tool,
)
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")


## ഘട്ടം 1: ഘടനയുള്ള ഔട്ട്പുട്ടുകൾക്കായി പിഡാന്റിക് മോഡലുകൾ നിർവചിക്കുക

ഈ മോഡലുകൾ ഏജന്റുകൾ തിരികെ നൽകുന്ന **സ്കീമ** നിർവചിക്കുന്നു. പിഡാന്റിക് ഉപയോഗിച്ച് `response_format` ഉപയോഗിക്കുന്നത് ഉറപ്പു വരുത്തുന്നു:
- ✅ തര-സുരക്ഷിതമായ ഡാറ്റ എക്‌സ്‌ട്രാക്ഷൻ
- ✅ സ്വയം ക്രമീകരണ പരിശോധന
- ✅ സ്വതന്ത്രരൂപത്തിലെ പ്രതികരണങ്ങളിൽ നിന്നുള്ള പാഴ്‌സിംഗ് പിഴവുകൾ ഇല്ലാതാക്കുന്നു
- ✅ ഫീൽഡുകൾ അടിസ്ഥാനമാക്കി എളുപ്പത്തിലുള്ള നിബന്ധനാപരമായ റൂട്ടിംഗ്


In [ ]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

## പടി 2: ഹോട്ടല്‍ ബുക്കിംഗ് ടൂൾ സൃഷ്ടിക്കുക

ഈ ടൂൾ **availability_agent** റൂമുകള്‍ ലഭ്യമാണ് എന്നു പരിശോധിക്കാൻ വിളിക്കും. `@ai_function` ഡെക്കറേറ്റർ ഉപയോഗിക്കുന്നത്:
- ഒരു Python ഫംഗ്ഷൻ AI-കോളബിളായ ടൂളായി പരിവർത്തനം ചെയ്യാൻ
- LLM ഉം JSON സ്കീമയും സ്വയം സൃഷ്ടിക്കാൻ
- പാരാമീറ്റർ സാധുത പരിശോധന കൈകാര്യം ചെയ്യാൻ
- ഏജന്റുകൾ സ്വയം വിളിക്കാനും സജ്ജമാക്കാൻ

ഈ ഡെമോയ്ക്ക്:
- **സ്റ്റോക്ക്ഹോം, സിയാറ്റിൽ, ടോക്യോ, ലണ്ടൻ, ആംസ്റ്റർഡാം** → റൂമുകൾ ഉണ്ട് ✅
- **മറ്റു എല്ലാ നഗരങ്ങളും** → റൂമുകൾ ഇല്ല ❌


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.

    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

## ഘട്ടം 3: റൂട്ടിങ്ങിനുള്ള അവസ്ഥഫങ്ഷനുകൾ നിർവചിക്കുക

ഈ ഫങ്ഷനുകൾ ഏജന്റിന്റെ പ്രതികരണം പരിശോധിച്ച് വർക്‌ഫ്ലോയിൽ എങ്ങനെ ദിശാബോധനം ചെയ്യണമെന്നത് നിശ്ചയിക്കുന്നു.

**പ്രധാന പാറ്റേൺ:**
1. സന്ദേശം `AgentExecutorResponse` ആണോ എന്ന് പരിശോധിക്കുക
2. ഘടനയുള്ള ഔട്ട്പുട്ട് (Pydantic മോഡൽ) പാഴ്‌സ് ചെയ്യുക
3. റൂട്ടിംഗ് നിയന്ത്രിക്കുന്നതിനായി `True` അല്ലെങ്കിൽ `False` മടക്കി നൽകുക

അടുത്ത എക്സിക്യൂട്ടറെ വിളിക്കാന്‍ വേണമെന്നത് തീരുമാനിക്കുന്നതിന് ഈ അവസ്ഥകൾ **എഡ്ജുകൾ** ൽ വർക്‌ഫ്ലോ വിലയിരുത്തും.


In [ ]:
def has_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels ARE available.
    
    Returns True if the destination has hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return True  # Default to True if unexpected type

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels are NOT available.
    
    Returns True if the destination has no hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception as e:
        return False


print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")

## പടിവരു 4: കസ്റ്റം ഡിസ്പ്ലേ എക്സിക്യൂട്ടർ സൃഷ്ടിക്കുക

എക്സിക്യൂട്ടറുകൾ ട്രാൻസ്ഫോർമേഷനുകൾ അല്ലെങ്കിൽ സൈഡ് എഫക്ടുകൾ നടത്തുന്ന വർക്ക്‌ഫ്ലോ_COMPONENT_സാണ്. ഫൈനൽ ഫലം ഡിസ്പ്ലേ ചെയ്യാൻ `@executor` ഡെക്കറേറ്റർ ഉപയോഗിച്ച് ഒരു കസ്റ്റം എക്സിക്യൂട്ടർ സൃഷ്ടിക്കുന്നു.

**പ്രധാന ആശയങ്ങൾ:**
- `@executor(id="...")` - ഒരു ഫങ്ഷൻ വർക്ക്‌ഫ്ലോ എക്സിക്യൂട്ടറായി രജിസ്ട്രർ ചെയ്യുന്നു
- `WorkflowContext[Never, str]` - ഇൻപുട്ട്/ഔട്ട്പുട്ടിനുള്ള ടൈപ്പ് നിർദ്ദേശങ്ങൾ
- `ctx.yield_output(...)` - ഫൈനൽ വർക്ക്‌ഫ്ലോ ഫലം യീൽഡ് ചെയ്യുന്നു


In [ ]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created with @executor decorator")

## പടി 5: പരിതസ്ഥിതി വേരിയബിളുകൾ ലോഡ് ചെയ്യുക

LLM ക്ലയന്റിനെ സജ്ജമാക്കുക. ഈ ഉദാഹരണം താഴെപ്പറയുന്നവയുമായി പ്രവർത്തിക്കുന്നു:
- **Microsoft Foundry** / **Azure OpenAI** (Responses API — ശിപാർശ ചെയ്‌തിരിക്കുന്നു)
- **Azure OpenAI**
- **OpenAI**


In [ ]:
# Load environment variables
load_dotenv()

# Configure the Microsoft Foundry provider with keyless authentication
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)


## ഘട്ടം 6: ഘടനാ നിഷ്പന്നങ്ങളോടുള്ള AI ഏജന്റുകളെ സൃഷ്ടിക്കുക

ഞങ്ങൾ **മൂന്നു വിശേഷ ഷ്‌ടെഡ് ഏജന്റുകളെ** സൃഷ്ടിക്കുന്നു, ഓരോന്നും `AgentExecutor`-ൽ മൂടപ്പെട്ടിരിക്കുന്നു:

1. **availability_agent** - ടൂൾ ഉപയോഗിച്ച് ഹോട്ടൽ ലഭ്യത പരിശോധിക്കുന്നു
2. **alternative_agent** - മറ്റാന്റുള്ള നഗരങ്ങൾ നിർദ്ദേശിക്കുന്നു (റൂമുകൾ ഇല്ലാതെ)
3. **booking_agent** - ബുക്കിംഗ് പ്രേരിപ്പിക്കുന്നു (റൂമുകൾ ലഭ്യമായപ്പോള്)

**പ്രധാന ഫീച്ചറുകൾ:**
- `tools=[hotel_booking]` - ഏജന്റിനായി ടൂൾ നൽകുന്നു
- `response_format=PydanticModel` - ഘടനയിൽ ഉള്ള JSON ഔട്ട്പുട്ട് നിർബന്ധിതമാക്കുന്നു
- `AgentExecutor(..., id="...")` - വാർക്ക്‌ഫ്ലോ ഉപയോഗത്തിനായി ഏജന്റ് മൂടുന്നു


In [ ]:
# Agent 1: Check availability with tool
availability_agent = AgentExecutor(
    provider.as_agent(
        name="availability-agent",
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    provider.as_agent(
        name="alternative-agent",
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    provider.as_agent(
        name="booking-agent",
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)


## ഘട്ടം 7: സസ്യ മേഖലകളുള്ള പ്രവർത്തനപ്രക്രിയ നിർമാണം

ഇപ്പോൾ നാം `WorkflowBuilder` ഉപയോഗിച്ച് സസ്യ മാർഗനിർദ്ദേശത്തോടെ ഗ്രാഫ് നിർമ്മിക്കുന്നു:

**പ്രവർത്തനപ്രക്രിയ ഘടന:**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙         ↘
[no_availability]  [has_availability]
        ↓              ↓
alternative_agent  booking_agent
        ↓              ↓
    display_result ←───┘
```

**പ്രധാന മാർഗ്ഗങ്ങൾ:**
- `.set_start_executor(...)` - പ്രവേശന പോയിന്റ് സജ്ജമാക്കുന്നു
- `.add_edge(from, to, condition=...)` - സസ്യമായ സന്ധി കൂട്ടുന്നു
- `.build()` - പ്രവർത്തനപരിപാടി സമാപിപ്പിക്കുന്നു


In [ ]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing:</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result
        </p>
    </div>
""")
)

## ഘട്ടം 8: ടെസ്റ്റ് കേസ് 1 ഓടിക്കുക - ലഭ്യത ഇല്ലാത്ത നഗരം (പാരിസ്)

നമ്മുടെ സിമുലേഷനിൽ ഏതൊരു മുറികളും ലഭ്യമല്ലാത്ത പാരിസ് എന്ന സ്‌ഥലത്തുള്ള ഹോട്ടലുകൾ തേടി **ലഭ്യതയില്ലാത്ത** പാത പരിശോധിക്കാം.


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → alternative_agent → display_result</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Paris"])], should_respond=True
)

# Run the workflow
events_paris = await workflow.run(request_paris)
outputs_paris = events_paris.get_outputs()

# Display results
if outputs_paris:
    result_paris = AlternativeResult.model_validate_json(outputs_paris[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT (Paris)</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_paris.alternative_destination}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_paris.reason}</p>
            </div>
        </div>
    """)
    )

## ഘട്ടം 9: ടെസ്റ്റ് കേസ് 2 - നഗരത്തിനൊപ്പം ലഭ്യത (സ്റ്റോക്ക്‌ഹോം)

ഇപ്പോൾ സ്റ്റോക്ക്‌ഹോമിലെ ഹോട്ടലുകൾ (ഞങ്ങളുടെ സിമുലേഷനിൽ റൂമുകൾ ഉള്ള) ആവശ്യപ്പെടുന്നതിനിലൂടെ **ലഭ്യതാ** പാത പരീക്ഷിക്കാം.


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Stockholm"])], should_respond=True
)

# Run the workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
            </div>
        </div>
    """)
    )

## പ്രധാന പഠിപ്പുകളും അടുത്ത പടികളും

### ✅ നിങ്ങൾ പഠിച്ചതു്:

1. **WorkflowBuilder പാറ്റേൺ**
   - പ്രവേശന ബിന്ദു നിർവ്വചിക്കാൻ `.set_start_executor()` ഉപയോഗിക്കുക
   - ക്ഷിതിജ റൂട്ടിംഗ് ചെയ്യാൻ `.add_edge(from, to, condition=...)` ഉപയോഗിക്കുക
   - വർക്‌ഫ്ലോ അവസാനിപ്പിക്കാൻ `.build()` വിളിക്കുക

2. **ക്ഷിതിജ റൂട്ടിംഗ്**
   - Condition ഫംഗ്ഷനുകൾ `AgentExecutorResponse` പരിശോധിക്കുന്നു
   - റൂട്ടിംഗ് തീരുമാനങ്ങൾ എടുക്കാൻ ഘടനാപരമായ ഔട്ട്പുട്ടുകൾ പാഴ്സുചെയ്യുക
   - ഒരു എഡ്ജ് പ്രവർത്തിപ്പിക്കാൻ `True` തിരിച്ചുകൊടുക്കുക, അനുയോജ്യമായിട്ടില്ലെങ്കിൽ `False`

3. **ടൂൾ ഇന്റഗ്രേഷൻ**
   - Python ഫംഗ്ഷനുകൾ AI ടൂളുകളായി മാറ്റാൻ `@ai_function` ഉപയോഗിക്കുക
   - ഏജന്റുകൾ ആവശ്യമായപ്പോൾ ടൂളുകൾ ഓട്ടോമാറ്റിക്കായി വിളിക്കുന്നു
   - ടൂളുകൾ JSON തിരികെ നൽകുന്നു, ഏജന്റുകൾ അതു പാഴ്സുചെയ്യുന്നു

4. **ഘടനാപരമായ ഔട്ട്പുട്ടുകൾ**
   - ടൈപ്പ്-സേഫ് ഡേറ്റാ എക്സ്ട്രാക്ഷനായി Pydantic മോഡലുകൾ ഉപയോഗിക്കുക
   - ഏജന്റുകൾ സൃഷ്ടിക്കുമ്പോൾ `response_format=MyModel` সেট് ചെയ്യുക
   - പ്രതികരണങ്ങൾ `Model.model_validate_json()` ഉപയോഗിച്ച് പാഴ്സുചെയ്യുക

5. **കസ്റ്റ്​ം എക്സിക്യൂട്ടറുകൾ**
   - വർക്‌ഫ്ലോ ഘടകങ്ങൾ സൃഷ്ടിക്കാൻ `@executor(id="...")` ഉപയോഗിക്കുക
   - എക്സിക്യൂട്ടറുകൾ ഡേറ്റ മാറ്റാം അല്ലെങ്കിൽ സൈഡ് ഇഫക്റ്റ്സ് നടത്താം
   - വർക്‌ഫ്ലോ ഫലം ഉൽപ്പാദിപ്പിക്കാൻ `ctx.yield_output()` ഉപയോഗിക്കുക

### 🚀 ആധുനിക ലോക ഉപയോഗങ്ങൾ:

- **ട്രാവൽ ബുക്കിംഗ്**: ലഭ്യത പരിശോധിക്കുക, പകരം നിർദ്ദേശിക്കുക, ഓപ്ഷനുകൾ താരതമ്യ ചെയ്യുക
- **കസ്റ്റമർ സർവീസ്**: പ്രശ്ന തരം, മനോഭാവം, മുൻഗണന അടിസ്ഥാനമാക്കി റൂട്ടിംഗ്
- **ഇ-കൊമേഴ്സ്**: ഇൻവെന്ററി പരിശോധിക്കുക, പകരം നിർദ്ദേശിക്കുക, ഓർഡറുകൾ പ്രോസസ്സ് ചെയ്യുക
- **കണ്ടന്റ് മോഡറേഷൻ**: ടോക്സിസിറ്റി സ്കോറുകൾ, ഉപയോക്തൃ ഫ്ലാഗുകൾ പ്രകാരം റൂട്ടിംഗ്
- **അപ്രൂവൽ വർക്‌ഫ്ലോകൾ**: തുക, ഉപയോക്തൃ സ്ഥാനമാന, റിസ്ക് നില അടിസ്ഥാനമാക്കി റൂട്ടിംഗ്
- **മൾട്ടി-സ്റ്റെജ് പ്രോസസ്സ്**: ഡേറ്റാ നിലവാരം, പൂർണത അടിസ്ഥാനമാക്കി റൂട്ടിംഗ്

### 📚 അടുത്ത പടികൾ:

- കൂടുതൽ സങ്കീൺ വ്യവസ്ഥകൾ ചേർക്കുക (മൾട്ടിപ്പിൾ ക്രൈറ്റീരിയ)
- വർക്‌ഫ്ലോ സ്റ്റേറ്റ് മാനേജ്മെന്റോടെ ലൂപ്പുകൾ നടപ്പിലാക്കുക
- പുനരുപയോഗിക്കാവുന്ന ഘടകങ്ങൾക്ക് സബ്വർക്‌ഫ്ലോകൾ ചേർക്കുക
- യഥാർത്ഥ API-കളുമായി ഇന്റഗ്രേറ്റ് ചെയ്യുക (ഹോട്ടൽ ബുക്കിംഗ്, ഇൻവെന്ററി സിസ്റ്റങ്ങൾ)
- പിശക് കൈകാര്യം ചെയ്യലും ഫാൾബാക്ക് പാതകളും ചേർക്കുക
- ഇൻബിൽറ്റ് വിസ്വലൈസേഷന് ടൂളുകൾ ഉപയോഗിച്ച് വർക്‌ഫ്ലോകൾ വിശ്വലൈസ് ചെയ്യുക


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**അറിയിപ്പ്**:
ഈ രേഖ AI പരിഭാഷാ സേവനം [Co-op Translator](https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച് പരിഭാഷപ്പെടുത്തിയതാണ്. ഞങ്ങൾ കൃത്യതയ്ക്കായി ശ്രമിക്കുന്നുവെങ്കിലും, ഓട്ടോമേറ്റഡ് പരിഭാഷകളിൽ പിഴവുകൾ അല്ലെങ്കിൽ തെറ്റായ വിവരങ്ങൾ ഉണ്ടാകാൻ സാധ്യതയുണ്ട്. അതിന്റെ സ്വാഭാവിക ഭാഷയിലുള്ള അസൽ രേഖയാണ് പ്രാമാണികമായ ഉറവിടമായി പരിഗണിക്കേണ്ടത്. നിർണായകമായ വിവരങ്ങൾക്ക്, പ്രൊഫഷണൽ മനുഷ്യ പരിഭാഷ ശുപാർശ ചെയ്യുന്നു. ഈ പരിഭാഷ ഉപയോഗിച്ച് ഉണ്ടാകുന്ന തെറ്റിദ്ധാരണകൾ അല്ലെങ്കിൽ തെറ്റായ വ്യാഖ്യാനങ്ങൾക്കായി ഞങ്ങൾ ഉത്തരവാദികളല്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
